# CV Pipeline — Image Classification from Scratch

**Dataset:** CIFAR-10 (torchvision auto-download, ~170 MB)  
**Mirrors:** Any Kaggle image classification task  

| Stage | What it covers |
|---|---|
| 1 | Data loading + augmentation transforms |
| 2 | Patch/sub-image extraction (competition hint) |
| 3 | 2D CNN from scratch (ConvBlock stack) |
| 4 | Training loop with CosineAnnealingLR |
| 5 | TTA inference (free accuracy boost) |

**Rule:** No pretrained models. Build Conv->BN->ReLU->Pool blocks from scratch.

## Stage 0 — Config

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision
import torchvision.transforms as T
import warnings
warnings.filterwarnings('ignore')

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 128
EPOCHS      = 15
LR          = 1e-3
N_CLASSES   = 10
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f'Device  : {DEVICE}')
print(f'Classes : {CLASS_NAMES}')

---
## Stage 1 — Data Loading + Transforms

### Why augmentation at train time only?

The model should be **invariant** to:
- Horizontal flips (a cat facing left = a cat facing right)
- Small translations (random crop with padding)
- Brightness / contrast shifts (same object, different lighting)

We apply these **only at train time** — random perturbations force the model to learn more robust features instead of memorizing exact pixel positions.

At test time we use the **clean, deterministic** transform (just normalize).

### Normalization
`(pixel - mean) / std` per channel. Brings inputs to roughly N(0,1).  
These stats are precomputed from the CIFAR-10 training set.

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)  # per-channel mean of CIFAR-10 train set
STD  = (0.2470, 0.2435, 0.2616)

# Train transform: augment then normalize
train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),       # flip with 50% chance
    T.RandomCrop(32, padding=4),         # pad 4px on each side, then random 32x32 crop
    T.ColorJitter(brightness=0.2,        # random brightness/contrast/saturation
                  contrast=0.2,
                  saturation=0.2),
    T.ToTensor(),                        # PIL HWC uint8 [0,255] -> CHW float32 [0,1]
    T.Normalize(MEAN, STD),              # (x - mean) / std per channel
])

# Test transform: only normalize, no randomness
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_raw = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=train_transform)
test_raw  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=test_transform)

train_loader = DataLoader(train_raw, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
test_loader  = DataLoader(test_raw,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train samples : {len(train_raw):,}')
print(f'Test  samples : {len(test_raw):,}')
print(f'Train batches : {len(train_loader)}')

xb, yb = next(iter(train_loader))
print(f'\nBatch X shape : {xb.shape}  -> (batch, channels, H, W)')
print(f'Pixel range (after normalize): [{xb.min():.2f}, {xb.max():.2f}]')

---
## Stage 2 — Patch Extraction (sub-image cropping)

The kisi-kisi says: **"sub-image / patch cropping"** — relevant when images are large (512x512+).

**Strategy for large-image tasks:**
1. Slide a window over the image -> collect patches (e.g. 64x64)
2. Classify each patch independently with the CNN
3. Aggregate patch predictions -> image-level prediction (majority vote / avg probs)

CIFAR-10 is already 32x32, so we demo the function here — paste it on competition day when images are larger.

In [ ]:
def extract_patches(img_chw: torch.Tensor, patch_size: int = 16, stride: int = 8):
    """
    Slide a window over a single image and return all patches.

    img_chw  : (C, H, W) float tensor
    Returns  : (N_patches, C, patch_size, patch_size)

    Example: 32x32 image, patch=16, stride=8
      x positions: 0, 8, 16  -> 3 positions
      y positions: 0, 8, 16  -> 3 positions
      total = 9 patches
    """
    C, H, W  = img_chw.shape
    patches  = []
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patches.append(img_chw[:, y:y+patch_size, x:x+patch_size])
    return torch.stack(patches)  # (N, C, ph, pw)


sample_img = xb[0]   # (3, 32, 32) -- one image from the batch
patches    = extract_patches(sample_img, patch_size=16, stride=8)

print(f'Image shape   : {sample_img.shape}')
print(f'Patches shape : {patches.shape}  ({patches.shape[0]} patches of 16x16)')
print()
print('On competition day with 512x512 images, patch=64, stride=32:')
fake_large = torch.zeros(3, 512, 512)
p = extract_patches(fake_large, patch_size=64, stride=32)
print(f'  -> {p.shape[0]} patches per image')

---
## Stage 3 — 2D CNN Architecture

### ConvBlock
```
Conv2d  -- learnable 3x3 filters slide over spatial dims (same-padding: size preserved)
  |
BatchNorm2d  -- normalizes each channel's activations -> stable training, faster convergence
  |
ReLU  -- non-linearity: max(0, x). Kills negatives.
  |
MaxPool2d(2)  -- takes max in each 2x2 window -> halves H and W
```

### How spatial dims change
```
Input : (B,  3, 32, 32)
Block1: (B, 32, 16, 16)   <- 32 filters, MaxPool halves to 16
Block2: (B, 64,  8,  8)   <- 64 filters, MaxPool halves to 8
Block3: (B,128,  4,  4)   <- 128 filters
Block4: (B,256,  4,  4)   <- 256 filters, NO pool (preserve spatial)
AvgPool:(B,256,  2,  2)   <- AdaptiveAvgPool to fixed 2x2
Flatten:(B,1024)           <- 256*2*2 = 1024
Linear :(B,512) -> ReLU -> Dropout
Linear :(B, 10)            <- raw logits
```

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel: int = 3, pool: bool = True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel,
                              padding=kernel // 2,  # same-padding: output size = input size
                              bias=False)            # bias=False when followed by BN
        self.bn   = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x))))


class SimpleCNN(nn.Module):
    def __init__(self, n_classes: int = 10, in_channels: int = 3):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),       # (B,3,32,32)  -> (B,32,16,16)
            ConvBlock(32, 64),                # (B,32,16,16) -> (B,64,8,8)
            ConvBlock(64, 128),               # (B,64,8,8)   -> (B,128,4,4)
            ConvBlock(128, 256, pool=False),  # (B,128,4,4)  -> (B,256,4,4) -- no pool
            nn.AdaptiveAvgPool2d((2, 2)),     # -> (B,256,2,2) -- handles any input size!
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                     # (B,256,2,2) -> (B,1024)
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.5),                  # randomly zero 50% of neurons -> regularization
            nn.Linear(512, n_classes),        # -> (B,10) raw logits
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = SimpleCNN(n_classes=N_CLASSES).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

# Verify forward pass dimensions
with torch.no_grad():
    dummy = torch.zeros(4, 3, 32, 32).to(DEVICE)
    out   = model(dummy)
    print(f'Forward check: {dummy.shape} -> {out.shape}  (should be [4, {N_CLASSES}])')

---
## Stage 4 — Training

- **Loss:** `CrossEntropyLoss` = `-log(softmax(logit)[true_class])`
- **Optimizer:** Adam with `weight_decay=1e-4` (L2 regularization)
- **Scheduler:** `CosineAnnealingLR` — smoothly decays LR from `1e-3` to ~0 over all epochs. Better than step-decay.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0

print('{:>5} | {:>8} | {:>8} | {:>9} | {:>7}'.format('Epoch', 'LR', 'Loss', 'TrainAcc', 'ValAcc'))
print('-' * 48)

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = correct = total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()          # clear old gradients
        logits = model(xb)             # forward pass: (B, 10)
        loss   = criterion(logits, yb) # scalar loss
        loss.backward()                # backprop: compute d_loss / d_param for all params
        optimizer.step()               # gradient descent: param -= lr * grad

        total_loss += loss.item() * len(yb)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += len(yb)

    train_loss = total_loss / total
    train_acc  = correct / total

    # --- Validate ---
    model.eval()
    val_correct = val_total = 0
    with torch.no_grad():              # no gradients at test time (saves memory + compute)
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_correct += (model(xb).argmax(1) == yb).sum().item()
            val_total   += len(yb)
    val_acc = val_correct / val_total

    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    flag = ' <- best' if val_acc > best_val_acc else ''
    print('{:>5} | {:>8.5f} | {:>8.4f} | {:>9.4f} | {:>7.4f}{}'.format(
        epoch, current_lr, train_loss, train_acc, val_acc, flag))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn.pth')

print('\nBest validation accuracy: {:.4f}'.format(best_val_acc))

---
## Stage 5 — Test-Time Augmentation (TTA)

**What it is:** Run inference N times, each with a different random augmentation. Average the probability outputs.

**Why it works:** Each augmented view gives a slightly different probability vector. Averaging reduces variance — the model is less likely to be confidently wrong on any single view.

**Cost:** N x inference time (use N=5).  
**Gain:** typically +0.5–2% accuracy for free.

In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load('best_cnn.pth', map_location=DEVICE))
model.eval()

# TTA uses the same augmentations as training
tta_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def predict_tta(model, n_tta: int = 5, batch_size: int = 256):
    """
    Runs n_tta augmented passes and averages softmax probabilities.
    Returns (N_samples, N_classes) probability matrix.
    """
    all_probs = []
    tta_ds     = torchvision.datasets.CIFAR10(root='./data', train=False,
                                              download=False, transform=tta_transform)
    tta_loader = DataLoader(tta_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    for run in range(n_tta):
        run_probs = []
        with torch.no_grad():
            for xb, _ in tta_loader:
                xb    = xb.to(DEVICE)
                probs = torch.softmax(model(xb), dim=1).cpu().numpy()
                run_probs.append(probs)
        all_probs.append(np.vstack(run_probs))
        print(f'  TTA run {run+1}/{n_tta}')

    return np.mean(all_probs, axis=0)  # average over runs


print('Running TTA (5 passes)...')
tta_probs   = predict_tta(model, n_tta=5)
tta_preds   = tta_probs.argmax(axis=1)
true_labels = np.array(test_raw.targets)

tta_acc = (tta_preds == true_labels).mean()
print(f'\nSingle-pass best : {best_val_acc:.4f}')
print(f'TTA accuracy     : {tta_acc:.4f}')
print(f'TTA boost        : {tta_acc - best_val_acc:+.4f}')

In [ ]:
# Per-class accuracy breakdown
print('Per-class accuracy:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask    = true_labels == cls_idx
    cls_acc = (tta_preds[mask] == true_labels[mask]).mean()
    bar     = '#' * int(cls_acc * 30)
    print(f'  {cls_name:>12} : {bar:<30} {cls_acc:.3f}')

---
## Stage 6 — Submission Template

```python
import pandas as pd

# On Kaggle:
test_df = pd.read_csv('/kaggle/input/.../sample_submission.csv')

# Assign predictions
test_df['label'] = tta_preds                       # integer labels
# or if labels are strings:
# test_df['label'] = [CLASS_NAMES[p] for p in tta_preds]

# Safety checks
assert test_df['label'].isnull().sum() == 0
assert len(test_df) == len(test images)

test_df.to_csv('submission.csv', index=False)
```

In [ ]:
import pandas as pd

mock_sub = pd.DataFrame({
    'id'   : range(len(tta_preds)),
    'label': [CLASS_NAMES[p] for p in tta_preds],
})
print(mock_sub.head(10).to_string(index=False))
print(f'\nValue counts:\n{mock_sub["label"].value_counts()}')

---
## Key Competition Decisions

| Decision | Why |
|---|---|
| Normalize images | Pixel values to ~N(0,1) — stable gradients |
| `bias=False` in Conv before BN | BN's shift parameter makes bias redundant — saves params |
| `AdaptiveAvgPool` instead of fixed Flatten | Model works on any image resolution without changing code |
| CosineAnnealingLR | Smooth LR decay avoids abrupt drops that cause loss spikes |
| TTA at inference | Free +0.5–2% — always worth the extra few minutes |
| Patch extraction | For large images: classify patches -> aggregate |

### If accuracy is stuck
1. Add more augmentation (`RandomRotation`, `Cutout`)
2. Increase network depth (add one more `ConvBlock`)
3. Use `weight_decay=1e-3` (stronger L2)
4. Train longer with a lower final LR